In [4]:
from typing import Dict, Any
from typing import TypedDict,Any, Annotated, Sequence,Optional
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, SystemMessage,AnyMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END,START
from langgraph.prebuilt import ToolNode
import json
from langchain_core.messages import RemoveMessage
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image,display
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool
from langgraph.graph import MessagesState
from pydantic import BaseModel, Field
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from langchain_core.prompts import PromptTemplate
from typing import Literal
import pandas as pd
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
import matplotlib.pyplot as plt
import seaborn as sns
from langgraph.prebuilt import create_react_agent
from typing import TypedDict, Annotated, Any, Optional, Literal
from langgraph.graph import StateGraph
from langchain_core.messages import AnyMessage
import operator
import os
from langfuse import observe
import re
from typing import TypedDict, List, Optional
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from langchain.tools import tool
from langgraph.graph.message import add_messages 
from typing import TypedDict, Annotated, Optional, Literal, Dict, List, Any

Defining the State Schema of the Agent

In [43]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-3-haiku",  # Best for tool calling
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)
llm.invoke("Explain Langraph in one word")

AIMessage(content='Concise.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 15, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.25e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.25e-05, 'upstream_inference_prompt_cost': 3.75e-06, 'upstream_inference_completions_cost': 8.75e-06}}, 'model_provider': 'openai', 'model_name': 'anthropic/claude-3-haiku', 'system_fingerprint': None, 'id': 'gen-1774817025-nlPCvHy7Lstj9aUHPsC6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3b56-99d5-7a82-8456-c87c45ff6adf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 7, 'total_tokens': 22, 'input_token_details

In [89]:

import re
from typing import TypedDict, List, Optional
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
llm = ChatOpenAI(
    model="llama-3.1-8b-instant",
    api_key="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",
    base_url="https://api.groq.com/openai/v1",
    temperature=0
)
llm.invoke("Who is hitler")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = chromadb.PersistentClient(path="./chroma_db23", settings=Settings(anonymized_telemetry=False))
collection = client.get_collection("course_planner")
print(f"✅ ChromaDB ready | chunks: {collection.count()}")

class AgentState(TypedDict):
    student_query:      str
    completed_courses:  List[str]
    grades:             Optional[dict]
    target_program:     Optional[str]
    target_term:        Optional[str]
    max_credits:        Optional[int]
    catalog_year:       Optional[str]
    transfer_credits:   Optional[List[str]]
    missing_fields:         List[str]
    clarifying_questions:   List[str]
    needs_clarification:    bool
    retrieved_chunks:   List[dict]
    suggested_courses:  List[dict]
    risks_assumptions:  List[str]
    decision:   Optional[str]
    evidence:   List[str]
    next_step:  Optional[str]
    verification_passed:    bool
    unsupported_claims:     List[str]
    citation_coverage:      Optional[float]
    retry_count:            int
    is_not_in_catalog:      bool
    suggested_next_action:  Optional[str]
    answer:      str
    why:         str
    citations:   List[str]
    assumptions: List[str]
    prereq_results: List[dict]

def get_course_level(code):
    m = re.search(r'\.(\d+)', str(code))
    if m:
        num = re.sub(r'[A-Za-z]', '', m.group(1))
        return int(num) if num else 0
    return 0
def get_highest_completed_level(completed):
    levels = [get_course_level(c) for c in completed]
    return max(levels) if levels else 0
def is_lower_level_course(course_code, completed_courses):
    """Only skip truly introductory courses (6.100x, 6.1000) when student has advanced ones."""
    course_level = get_course_level(course_code)
    highest = get_highest_completed_level(completed_courses)
    # Only intro courses (level < 1010) get skipped, and only if student has 1010+
    return course_level < 1010 and highest >= 1010

def extract_course_code_from_chunk(text):
    m = re.search(r'Course Code[:\s]*(\d+\.\d+[A-Z]?\[?J?\]?)', text, re.I)
    return m.group(1).strip().replace("[J]","").replace("[","").replace("]","") if m else None
def extract_course_name_from_chunk(text):
    m = re.search(r'Course Name[:\s]*(.+)', text, re.I)
    return m.group(1).strip() if m else None
def extract_prereq_text(text):
    m = re.search(
        r'Prerequisite[s]?\s*:\s*\n?(.*?)(?:\n\s*\n|\nOffered|\nUnits|\nDescription|\nInstructor|\nSource|\nCo-?requisite|\Z)',
        text, re.DOTALL | re.I
    )
    return m.group(1).strip() if m else None
def check_prereqs(prereq_text, completed):
    if not prereq_text or prereq_text.lower() in ["none", "n/a", ""]:
        return True, [], "No prerequisites required"
    text = prereq_text.strip()
    completed_upper = [c.strip().upper() for c in completed]
    all_codes = re.findall(r'\d+\.\d+[A-Z]?', text, re.I)
    if not all_codes:
        return True, [], f"Non-course prerequisite: {text}"
    all_codes_upper = [c.upper() for c in all_codes]
    text_lower = text.lower()
    if " and " in text_lower and " or " not in text_lower:
        missing = [c for c in all_codes_upper if c not in completed_upper]
        if missing:
            return False, missing, f"AND: need ALL of {all_codes_upper}, missing {missing}"
        return True, [], f"AND: have ALL of {all_codes_upper} ✓"
    if " or " in text_lower and " and " not in text_lower:
        has_any = [c for c in all_codes_upper if c in completed_upper]
        if has_any:
            return True, [], f"OR: need ONE of {all_codes_upper}, have {has_any} ✓"
        return False, all_codes_upper, f"OR: need ONE of {all_codes_upper}, have NONE"
    or_parts = re.split(r'\s+or\s+', text_lower)
    for part in or_parts:
        part_clean = part.strip().strip('()')
        part_codes = [c.upper() for c in re.findall(r'\d+\.\d+[a-z]?', part_clean, re.I)]
        if not part_codes:
            continue
        if ' and ' in part_clean:
            if all(pc in completed_upper for pc in part_codes):
                return True, [], f"MIXED: OR→AND branch ({part_clean}) satisfied ✓"
        else:
            if part_codes[0] in completed_upper:
                return True, [], f"MIXED: OR branch ({part_codes[0]}) satisfied ✓"
    return False, all_codes_upper, f"MIXED: no branch satisfied from {or_parts}"
def run_prereq_checks(chunks, completed):
    results = []
    seen_codes = set()
    for chunk in chunks:
        text = chunk["text"]
        code = extract_course_code_from_chunk(text)
        if not code or code.upper() in seen_codes:
            continue
        seen_codes.add(code.upper())
        if code.upper() in [c.upper() for c in completed]:
            continue
        name = extract_course_name_from_chunk(text) or code
        prereq_text = extract_prereq_text(text)
        if prereq_text:
            eligible, missing, explanation = check_prereqs(prereq_text, completed)
        else:
            eligible, missing, explanation = True, [], "No prerequisites listed"
        skip_reason = None
        if is_lower_level_course(code, completed):
            skip_reason = f"Lower-level than completed (level {get_course_level(code)} < {get_highest_completed_level(completed)})"
        results.append({
            "course_code": code, "course_name": name,
            "prereq_text": prereq_text or "None",
            "eligible": eligible, "missing": missing,
            "explanation": explanation,
            "citation": chunk.get("citation", ""),
            "is_lower_level": skip_reason is not None,
            "skip_reason": skip_reason
        })
    return results

def retrieve_catalog_chunks(query: str, top_k=20) -> list:
    query_vec = embed_model.encode(
        [f"Represent this sentence: {query}"],
        normalize_embeddings=True
    ).tolist()
    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        score = round(1 - dist, 4)
        if score < 0.25:
            continue
        fname = meta.get("filename", "")
        cidx = meta.get("chunk_index", "")
        clean_name = fname.replace(".txt", "")
        citation = f"{clean_name} (chunk_{cidx})"
        chunks.append({
            "text": doc, "source": meta.get("source", ""),
            "category": meta.get("category", ""),
            "filename": fname, "chunk_id": cidx,
            "score": score, "citation": citation
        })
    return chunks
# ══════════════════════════════════════════════════════
# HELPER
# ══════════════════════════════════════════════════════
def _extract_section(text: str, section: str) -> str:
    escaped = re.escape(section.strip(":"))
    pattern = rf"(?i){escaped}\s*:?\s*(.*?)(?=\n[A-Z][^\n]{{2,}}:|\Z)"
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).strip() if match else ""
def extract_target_course(query):
    """Extract the specific course the student is asking about in prereq queries."""
    # "Can I take X", "enroll in X", "eligible for X"
    m = re.search(r'(?:take|enroll\s+in|eligible\s+for|register\s+for)\s+(\d+\.\d+[A-Z]?)', query, re.I)
    if m:
        return m.group(1).upper()
    # "Is X sufficient to enroll in Y" → target is Y
    m = re.search(r'(?:enroll\s+in|take)\s+(\d+\.\d+[A-Z]?)', query, re.I)
    if m:
        return m.group(1).upper()
    return None

# ══════════════════════════════════════════════════════
# AGENT 1 — INTAKE
# ══════════════════════════════════════════════════════
def intake_node(state: AgentState) -> AgentState:
    missing = []
    query_lower = state.get("student_query", "").lower()

    # Detect query type
    is_planning = any(w in query_lower for w in ["plan", "schedule", "next semester", "next term", "what courses", "what should i take"])
    is_prereq = any(w in query_lower for w in ["can i take", "eligible", "prerequisite", "prereq", "enroll", "sufficient"])
    is_program = any(w in query_lower for w in ["degree", "requirement", "elective", "hass", "units required", "laboratory", "either/or", "course 6-3", "6-3"])
    is_policy = any(w in query_lower for w in ["policy", "deadline", "waitlist", "override", "professor", "rating", "offered on", "add/drop"])

    # ONLY planning queries need full profile
    if is_planning:
        if not state.get("target_program"):
            missing.append("target_program")
        if state.get("completed_courses") is None:  # None = not provided. [] = no courses (valid)
            missing.append("completed_courses")
        if not state.get("target_term"):
            missing.append("target_term")
        if not state.get("max_credits"):
            missing.append("max_credits")

  

    if missing:
        prompt = f"""You are a student academic advisor. Student asked: {state['student_query']}
Missing: {missing}. Ask MAX {len(missing)} short clarifying questions."""
        response = llm.invoke(prompt)
        print(f"\n🔵 [INTAKE] Missing: {missing}")
        return {**state, "missing_fields": missing,
                "clarifying_questions": [response.content], "needs_clarification": True}

    print(f"\n🔵 [INTAKE] All fields present → proceeding")
    return {**state, "missing_fields": [], "clarifying_questions": [], "needs_clarification": False}

# ══════════════════════════════════════════════════════
# AGENT 2 — RETRIEVER (dedup + policy filter + coverage)
# ══════════════════════════════════════════════════════
def retriever_node(state: AgentState) -> AgentState:
    completed = state.get("completed_courses", [])
    query_lower = state.get("student_query", "").lower()
    is_planning = any(w in query_lower for w in ["plan","schedule","next semester","next term","what courses"])
    seen_ids = set()
    all_chunks = []
    def add_chunk(chunk):
        uid = f"{chunk['filename']}_{chunk['chunk_id']}"
        if uid in seen_ids:
            return
        seen_ids.add(uid)
        all_chunks.append(chunk)
    # Semantic search for each completed course
    for course in completed:
        for q in [f"prerequisite {course}", f"courses that require {course}", course]:
            for chunk in retrieve_catalog_chunks(q, top_k=20):
                add_chunk(chunk)
    # Search by student query
    for chunk in retrieve_catalog_chunks(state["student_query"], top_k=20):
        add_chunk(chunk)
    # For planning: explicitly retrieve program requirements
    if is_planning and state.get("target_program"):
        prog = state["target_program"]
        for q in [f"{prog} degree requirements", f"{prog} core courses",
                  f"{prog} required courses", f"program requirements {prog}"]:
            for chunk in retrieve_catalog_chunks(q, top_k=15):
                add_chunk(chunk)
    # Keyword fallback — full collection scan
    all_results = collection.get(include=["documents", "metadatas"])
    for doc, meta in zip(all_results["documents"], all_results["metadatas"]):
        filename = meta.get("filename", "")
        chunk_id = str(meta.get("chunk_index", ""))
        uid = f"{filename}_{chunk_id}"
        if uid in seen_ids:
            continue
        if any(c.lower() in doc.lower() for c in completed):
            clean_name = filename.replace(".txt", "")
            add_chunk({
                "text": doc, "source": meta.get("source", ""),
                "category": meta.get("category", ""),
                "filename": filename, "chunk_id": chunk_id,
                "score": 0.5, "citation": f"{clean_name} (chunk_{chunk_id})"
            })
    # Filter: for planning/prereq queries, drop policy chunks entirely
    if is_planning:
        filtered = [c for c in all_chunks if c.get("category","").lower() not in ["policy","policies"]]
    else:
        filtered = all_chunks
    print(f"\n🟡 [RETRIEVER] {len(filtered)} chunks (filtered from {len(all_chunks)}, deduped)")
    for c in filtered:
        print(f"  → [{c['category']}] {c['filename']} chunk_{c['chunk_id']} | score: {c['score']}")
    return {**state, "retrieved_chunks": filtered}
# ══════════════════════════════════════════════════════
# AGENT 3 — PLANNER (deterministic prereqs + skip intro + concise)
# ══════════════════════════════════════════════════════
def planner_node(state: AgentState) -> AgentState:
    completed = state.get("completed_courses", [])
    chunks = state.get("retrieved_chunks", [])
    query = state.get("student_query", "")
    query_lower = query.lower()

    prereq_results = run_prereq_checks(chunks, completed)

    recommended = [r for r in prereq_results if r["eligible"] and not r["is_lower_level"]]
    skipped_lower = [r for r in prereq_results if r["eligible"] and r["is_lower_level"]]
    not_eligible = [r for r in prereq_results if not r["eligible"]]

    # ── DETECT QUERY TYPE ──
    is_prereq = any(w in query_lower for w in ["can i take", "eligible", "prerequisite", "prereq", "enroll", "sufficient"])
    is_chain = any(w in query_lower for w in ["chain", "path", "trace", "before i can", "must i complete"])
    is_program = any(w in query_lower for w in ["how many", "elective", "requirement", "units required", "hass", "laboratory", "either/or", "6-3"])
    is_trick = any(w in query_lower for w in ["offered on", "best professor", "rating", "override", "deadline", "waitlist"])

    target_code = extract_target_course(query)
    target_result = None
    if target_code and is_prereq:  # Only use target for prereq queries, NOT chains
        for r in prereq_results:
            if r["course_code"].upper() == target_code:
                target_result = r
                break

    prereq_summary = ""
    if recommended:
        prereq_summary += "RECOMMENDED (eligible + appropriate level):\n"
        prereq_summary += "\n".join(f"  ✅ {r['course_code']} {r['course_name']} — {r['explanation']} [{r['citation']}]" for r in recommended)
    if skipped_lower:
        prereq_summary += "\n\nSKIPPED (lower-level intro courses):\n"
        prereq_summary += "\n".join(f"  ⬇️ {r['course_code']} {r['course_name']} — {r['skip_reason']}" for r in skipped_lower)
    if not_eligible:
        prereq_summary += "\n\nNOT ELIGIBLE (missing prereqs):\n"
        prereq_summary += "\n".join(f"  ❌ {r['course_code']} {r['course_name']} — Missing: {r['missing']} [{r['citation']}]" for r in not_eligible)

    context = "\n\n".join([
        f"[{c['citation']}]\n{c['text']}" for c in chunks
    ]) or "No relevant catalog chunks retrieved."

    grades_info = state.get('grades') or {}
    grades_str = ", ".join(f"{k}: {v}" for k, v in grades_info.items()) if grades_info else "Not provided"

    # ── DECISION LOGIC ──
        # ── DECISION LOGIC ──
    if is_prereq and target_code and target_result:
        if target_result["eligible"] and not target_result.get("is_lower_level"):
            correct_decision = "Eligible"
        else:
            correct_decision = "Not Eligible"
    elif is_prereq and target_code and not target_result:
        correct_decision = "Need More Info"
    else:
        # Chain / Program / Trick / Planning → based on whether any courses are available
        correct_decision = "Eligible" if recommended else ("Not Eligible" if not_eligible else "Need More Info")


    # ── BUILD PROMPT BASED ON QUERY TYPE ──
    if is_prereq and target_code and target_result and not target_result["eligible"]:
        # NOT ELIGIBLE — tell LLM explicitly what to write
        missing = target_result["missing"]
        focus = f"""The student asked about {target_code}. It is NOT ELIGIBLE.
Prerequisite: {target_result['prereq_text']}
Missing: {missing}
DO NOT put {target_code} in the Answer / Plan section.
Instead write: "You are NOT eligible for {target_code}."
Then explain what's missing and what to do next."""
    elif is_prereq and target_code and target_result and target_result["eligible"]:
        focus = f"The student asked about {target_code}. It IS eligible. Confirm eligibility with citation."
    elif is_chain:
        focus = f"""This is a PREREQUISITE CHAIN question. Show the full chain of prerequisites.
For each course in the chain, list its prerequisites from the catalog.
Format as: Course A → requires Course B → requires Course C.
Do NOT just say 'Eligible' or 'Not Eligible'. Show the full path."""
    elif is_program:
        focus = f"""This is a PROGRAM REQUIREMENT question. Answer the specific question asked.
Use the program documents in context to provide the exact answer with citations.
Do NOT list eligible courses. Just answer the question directly."""
    elif is_trick:
        focus = f"""Check if the catalog context contains the answer.
If NOT, say: "I don't have that information in the provided catalog/policies."
Then suggest: check the MIT Registrar website, schedule of classes, or contact your advisor."""
    else:
        focus = "List ALL recommended courses from the RECOMMENDED list."

    prompt = f"""You are a strict academic course planner. Be CONCISE.

Student Profile:
- Completed: {completed}
- Grades: {grades_str}
- Program: {state.get('target_program')}
- Term: {state.get('target_term')}

{focus}

═══ DETERMINISTIC RESULTS (GROUND TRUTH) ═══
{prereq_summary}
═════════════════════════════════════════════

Catalog Context:
{context}

RULES:
1. Do NOT recommend NOT ELIGIBLE courses in Answer / Plan
2. Do NOT invent grades not listed above
3. Cite every catalog fact as [course_name (chunk_id)]
4. If info not in catalog: "I don't have that information in the provided catalog/policies."

Output EXACTLY:

Answer / Plan:
<answer the question directly>

Why (requirements/prereqs satisfied):
<reasoning>

Citations:
<citations>

Assumptions / Not in catalog:
<unknowns>

DECISION: {correct_decision}

NEXT STEP:
<what to do next>"""

    response = llm.invoke(prompt)
    text = response.content

    # Force correct decision
    llm_decision = _extract_section(text, "DECISION").strip()
    if correct_decision != "N/A" and llm_decision.lower().replace(" ","") != correct_decision.lower().replace(" ",""):
        print(f"  ⚠️ [PLANNER] Overriding '{llm_decision}' → '{correct_decision}'")
        text = re.sub(r'(?i)DECISION\s*:\s*.*', f'DECISION: {correct_decision}', text)

    # For "Not Eligible" prereq queries — force clean answer if LLM still puts target in plan
    if is_prereq and correct_decision == "Not Eligible" and target_code:
        answer_section = _extract_section(text, "Answer / Plan")
        if target_code in answer_section and "not eligible" not in answer_section.lower() and "cannot" not in answer_section.lower():
            # LLM put the ineligible course in the plan — fix it
            text = re.sub(
                r'(?i)(Answer\s*/?\s*Plan\s*:)\s*\n?.*?(?=\n\s*Why|\n\s*\n)',
                f'\\1\nYou are NOT eligible for {target_code} {target_result["course_name"]}.\nMissing prerequisites: {", ".join(target_result["missing"])}',
                text, count=1
            )

    qtype = "prereq" if is_prereq else ("chain" if is_chain else ("program" if is_program else ("trick" if is_trick else "plan")))
    print(f"\n🟢 [PLANNER] {len(recommended)} recommended, {len(skipped_lower)} skipped, {len(not_eligible)} not eligible")
    print(f"  → Type: {qtype} | DECISION: {correct_decision}")

    return {
        **state,
        "answer": text, "prereq_results": prereq_results,
        "decision": correct_decision,
        "risks_assumptions": _extract_section(text, "Assumptions").split("\n"),
        "next_step": _extract_section(text, "NEXT STEP"),
        "why": _extract_section(text, "Why"),
        "citations": _extract_section(text, "Citations").split("\n"),
        "evidence": _extract_section(text, "Why").split("\n"),
        "assumptions": _extract_section(text, "Assumptions").split("\n"),
        "is_not_in_catalog": "not in catalog" in text.lower(),
        "suggested_next_action": _extract_section(text, "NEXT STEP"),
    }


# ══════════════════════════════════════════════════════
# AGENT 4 — VERIFIER (logic + subset + grade hallucination check)
# ══════════════════════════════════════════════════════
def verifier_node(state: AgentState) -> AgentState:
    prereq_results = state.get("prereq_results", [])
    answer = state.get("answer", "")
    completed = state.get("completed_courses", [])
    decision = state.get("decision", "")

    errors = []
    answer_section = _extract_section(answer, "Answer / Plan")

    eligible_set = {r["course_code"] for r in prereq_results if r["eligible"] and not r.get("is_lower_level")}
    not_eligible_set = {r["course_code"] for r in prereq_results if not r["eligible"]}
    lower_set = {r["course_code"] for r in prereq_results if r.get("is_lower_level")}
    completed_upper = {c.upper() for c in completed}

    # Only check course recommendations for Eligible/Not Eligible decisions
    if decision not in ["N/A", ""]:
        recommended_codes = re.findall(r'(\d+\.\d+[A-Z]?)', answer_section)
        for code in recommended_codes:
            if code.upper() in not_eligible_set or code in not_eligible_set:
                # Only flag if it seems like a recommendation, not a "you can't take this" statement
                surrounding = answer_section.lower()
                if "not eligible" not in surrounding and "cannot" not in surrounding and "missing" not in surrounding:
                    missing = next((r["missing"] for r in prereq_results if r["course_code"].upper() == code.upper()), [])
                    errors.append(f"NOT ELIGIBLE course {code} recommended in plan (missing: {missing})")
            elif code.upper() in lower_set or code in lower_set:
                errors.append(f"LOWER-LEVEL course {code} in plan")

    # Grade hallucination check
    grades = state.get("grades") or {}
    grade_mentions = re.findall(r'with (?:a |an )?(?:grade (?:of )?)?([A-F][+-]?)\b', answer, re.I)
    for g in grade_mentions:
        if not any(g.upper() in str(v).upper() for v in grades.values()):
            errors.append(f"Hallucinated grade '{g}'")

    verification_passed = len(errors) == 0

    if errors:
        print(f"\n🔴 [VERIFIER] FAILED — {len(errors)} issue(s):")
        for e in errors:
            print(f"    ✗ {e}")
    else:
        print(f"\n🔴 [VERIFIER] PASSED ✅")

    retry_count = state.get("retry_count", 0) + 1

    return {
        **state,
        "verification_passed": verification_passed,
        "unsupported_claims": errors,
        "citation_coverage": None,
        "retry_count": retry_count,
        "answer": state["answer"] if verification_passed else (
            state["answer"] + f"\n\n⚠️ VERIFICATION FAILED (attempt {retry_count}):\n" + "\n".join(errors)
        )
    }

# ══════════════════════════════════════════════════════
# GRAPH
# ══════════════════════════════════════════════════════
def should_clarify(state): return "end" if state["needs_clarification"] else "retriever"
def should_rewrite(state):
    if not state["verification_passed"] and state.get("retry_count", 0) < 2:
        print(f"   🔁 Retrying planner (attempt {state['retry_count']})")
        return "planner"
    return "end"
workflow = StateGraph(AgentState)
workflow.add_node("intake", intake_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("planner", planner_node)
workflow.add_node("verifier", verifier_node)
workflow.set_entry_point("intake")
workflow.add_conditional_edges("intake", should_clarify, {"end": END, "retriever": "retriever"})
workflow.add_edge("retriever", "planner")
workflow.add_edge("planner", "verifier")
workflow.add_conditional_edges("verifier", should_rewrite, {"planner": "planner", "end": END})
app = workflow.compile()
print("✅ Graph compiled\n")
# ══════════════════════════════════════════════════════
# TEST
# ══════════════════════════════════════════════════════
for step in app.stream({
    "student_query":        "What courses can I take next semester?",
    "completed_courses":    ["6.1000"],
    "grades":               {"6.1000": "A"},
    "target_program":       "Computer Science",
    "target_term":          "Fall",
    "max_credits":          48,
    "catalog_year":         "2024",
    "transfer_credits":     [],
    "missing_fields":       [],
    "clarifying_questions": [],
    "needs_clarification":  False,
    "retrieved_chunks":     [],
    "suggested_courses":    [],
    "risks_assumptions":    [],
    "decision":             "",
    "evidence":             [],
    "next_step":            "",
    "verification_passed":  False,
    "unsupported_claims":   [],
    "citation_coverage":    0.0,
    "retry_count":          0,
    "is_not_in_catalog":    False,
    "suggested_next_action":"",
    "answer":               "",
    "why":                  "",
    "citations":            [],
    "assumptions":          [],
    "prereq_results":       []
}):
    print("─── Node Output ───")
    for node, output in step.items():
        print(f"\n NODE: {node}")
        if output.get("answer"):
            print(output["answer"])
        elif output.get("clarifying_questions"):
            print(output["clarifying_questions"])
        elif "retrieved_chunks" in output:
            print(f"Retrieved {len(output['retrieved_chunks'])} chunks")
    print()

✅ ChromaDB ready | chunks: 36
✅ Graph compiled


🔵 [INTAKE] All fields present → proceeding
─── Node Output ───

 NODE: intake
Retrieved 0 chunks


🟡 [RETRIEVER] 25 chunks (filtered from 28, deduped)
  → [course] 6.1010 Fundamentals of Programming.txt chunk_0 | score: 0.7906
  → [course] 6.1000 Introduction to Programming and Computer Science.txt chunk_0 | score: 0.7884
  → [course] 6.1100 Computer Language Engineering.txt chunk_0 | score: 0.7779
  → [course] 6.1000 Introduction to Programming and Computer Science.txt chunk_1 | score: 0.7655
  → [course] 6.100A Introduction to Computer Science Programming in Python.txt chunk_0 | score: 0.7609
  → [course] 6.100L Introduction to Computer Science and Programming.txt chunk_0 | score: 0.7572
  → [course] 6.1020 Software Construction.txt chunk_0 | score: 0.7531
  → [course] 6.1120 Dynamic Computer Language Engineering.txt chunk_0 | score: 0.7469
  → [course] 6.1060 Software Performance Engineering.txt chunk_0 | score: 0.7433
  → [course] 6.

In [90]:
# ══════════════════════════════════════════════════════
# EVALUATION SUITE — 25 QUERIES + METRICS + 3 TRANSCRIPTS
# ══════════════════════════════════════════════════════
import time

def run_query(query, completed=None, major=None, term=None, max_credits=None, grades=None):
    state = {
        "student_query": query,
        "completed_courses": completed if completed is not None else [],  # [] is valid, not "missing"
        "grades": grades or {},
        "target_program": major or "Computer Science",  # default so intake never blocks
        "target_term": term or "Fall",
        "max_credits": max_credits or 4,
        "catalog_year": "2024",
        "transfer_credits": [],
        "missing_fields": [],
        "clarifying_questions": [],
        "needs_clarification": False,
        "retrieved_chunks": [],
        "suggested_courses": [],
        "risks_assumptions": [],
        "decision": "",
        "evidence": [],
        "next_step": "",
        "verification_passed": False,
        "unsupported_claims": [],
        "citation_coverage": 0.0,
        "retry_count": 0,
        "is_not_in_catalog": False,
        "suggested_next_action": "",
        "answer": "",
        "why": "",
        "citations": [],
        "assumptions": [],
        "prereq_results": []
    }
    result = app.invoke(state)
    return result


# ══════════════════════════════════════════════════════
# 25 TEST QUERIES
# ══════════════════════════════════════════════════════
test_suite = [
    # ── 10 PREREQUISITE CHECKS ──
    {"id": 1,  "cat": "prereq", "q": "Can I take 6.1010 if I passed 6.1000?",
     "completed": ["6.1000"], "expect": "Eligible",
     "rubric": "6.1010 prereq = 6.1000 OR (6.100A AND 6.100B). Student has 6.1000 → Eligible"},

    {"id": 2,  "cat": "prereq", "q": "I have taken 6.100A and 6.100B. Can I enroll in 6.1010?",
     "completed": ["6.100A", "6.100B"], "expect": "Eligible",
     "rubric": "6.1010 prereq = 6.1000 OR (6.100A AND 6.100B). Student has both → Eligible"},

    {"id": 3,  "cat": "prereq", "q": "Can I take 6.1010 without any prior courses?",
     "completed": [], "expect": "Not Eligible",
     "rubric": "6.1010 prereq = 6.1000 OR (6.100A AND 6.100B). Student has none → Not Eligible"},

    {"id": 4,  "cat": "prereq", "q": "I completed only 6.100A but not 6.100B. Am I eligible for 6.1010?",
     "completed": ["6.100A"], "expect": "Not Eligible",
     "rubric": "AND branch needs BOTH 6.100A+6.100B. Only has 6.100A. OR branch needs 6.1000. → Not Eligible"},

    {"id": 5,  "cat": "prereq", "q": "I have 6.1010 completed. Am I eligible for 6.1020 Software Construction?",
     "completed": ["6.1010"], "expect": "Eligible",
     "rubric": "6.1020 prereq = 6.1010. Student has it → Eligible"},

    {"id": 6,  "cat": "prereq", "q": "Can I take 6.1100 if I only have 6.1020?",
     "completed": ["6.1020"], "expect": "Not Eligible",
     "rubric": "6.1100 prereq = 6.1020 AND 6.1910. Missing 6.1910 → Not Eligible"},

    {"id": 7,  "cat": "prereq", "q": "I have 6.1020 and 6.1910. Can I take 6.1100?",
     "completed": ["6.1020", "6.1910"], "expect": "Eligible",
     "rubric": "6.1100 prereq = 6.1020 AND 6.1910. Has both → Eligible"},

    {"id": 8,  "cat": "prereq", "q": "Is 6.100A alone sufficient to enroll in 6.1010?",
     "completed": ["6.100A"], "expect": "Not Eligible",
     "rubric": "OR branch needs 6.1000 (missing). AND branch needs 6.100A+6.100B (missing B) → Not Eligible"},

    {"id": 9,  "cat": "prereq", "q": "Can I take 6.1120 if I have completed 6.1020?",
     "completed": ["6.1020"], "expect": "Eligible",
     "rubric": "6.1120 prereq = 6.1020 OR 6.1910. Student has 6.1020 → Eligible"},

    {"id": 10, "cat": "prereq", "q": "Can I take 6.1120 without any courses completed?",
     "completed": [], "expect": "Not Eligible",
     "rubric": "6.1120 prereq = 6.1020 OR 6.1910. Student has neither → Not Eligible"},

    # ── 5 MULTI-HOP PREREQUISITE CHAINS ──
    {"id": 11, "cat": "chain", "q": "What is the full prerequisite chain to reach 6.1020?",
     "completed": [], "expect": "chain",
     "rubric": "6.1020 needs 6.1010. 6.1010 needs 6.1000 or (6.100A+6.100B). Must show 2-step chain."},

    {"id": 12, "cat": "chain", "q": "Starting from zero, what path do I take to eventually take 6.1100?",
     "completed": [], "expect": "chain",
     "rubric": "6.1100 needs 6.1020+6.1910. 6.1020 needs 6.1010. 6.1010 needs 6.1000. Multi-hop."},

    {"id": 13, "cat": "chain", "q": "I have only completed 6.1000. Trace what courses I can eventually reach in CS.",
     "completed": ["6.1000"], "expect": "chain",
     "rubric": "6.1000 → 6.1010 → 6.1020 → 6.1040/6.1060/6.1100/6.1120. Must show chain progression."},

    {"id": 14, "cat": "chain", "q": "What courses must I complete before I can take 6.1040?",
     "completed": [], "expect": "chain",
     "rubric": "6.1040 prereq = 6.1020. 6.1020 prereq = 6.1010. 6.1010 prereq = 6.1000. Show full chain."},

    {"id": 15, "cat": "chain", "q": "I have completed 6.1010 and 6.1200. What next courses can I take and what do they unlock?",
     "completed": ["6.1010", "6.1200"], "expect": "chain",
     "rubric": "From 6.1010: can take 6.1020. From 6.1020: unlocks 6.1040/6.1060/6.1100/6.1120. Show progression."},

    # ── 5 PROGRAM REQUIREMENT QUESTIONS ──
    {"id": 16, "cat": "program", "q": "How many elective subjects do I need for the Course 6-3 CS degree?",
     "completed": [], "expect": "program",
     "rubric": "2 CS track + 2 CS/AI/EE track + 1 additional = 5 electives total. Must cite program doc."},

    {"id": 17, "cat": "program", "q": "What are the either/or options in the core CS requirements for Course 6-3?",
     "completed": [], "expect": "program",
     "rubric": "6.1400 or 6.1220, and 6.1800 or 6.1810 or 6.5831. Must list both OR groups."},

    {"id": 18, "cat": "program", "q": "What is the total units required for a BS in Computer Science at MIT?",
     "completed": [], "expect": "program",
     "rubric": "180-198 units total. 177 in major. Must cite program doc."},

    {"id": 19, "cat": "program", "q": "What is the laboratory requirement for the Course 6-3 CS degree?",
     "completed": [], "expect": "program",
     "rubric": "1 subject: 6.1010 Fundamentals of Programming. Must cite program doc."},

    {"id": 20, "cat": "program", "q": "How many HASS subjects are required for the MIT CS degree?",
     "completed": [], "expect": "program",
     "rubric": "8 HASS subjects required under GIR. Must cite program doc."},

    # ── 5 NOT-IN-DOCS / TRICK QUESTIONS ──
    {"id": 21, "cat": "trick", "q": "Is 6.1010 offered on Mondays and Wednesdays or Tuesdays and Thursdays?",
     "completed": [], "expect": "abstain",
     "rubric": "Schedule/day info not in catalog. Must abstain and suggest checking schedule of classes."},

    {"id": 22, "cat": "trick", "q": "Who is the best professor for 6.1010 based on student ratings?",
     "completed": [], "expect": "abstain",
     "rubric": "Ratings/rankings not in catalog. Must abstain. May list known instructors but not rank them."},

    {"id": 23, "cat": "trick", "q": "Can Professor Smith give me a prerequisite override for 6.1010?",
     "completed": [], "expect": "abstain",
     "rubric": "Override policies not in catalog. Must abstain and suggest contacting department/advisor."},

    {"id": 24, "cat": "trick", "q": "What is the exact add/drop deadline for Fall 2025?",
     "completed": [], "expect": "abstain",
     "rubric": "Specific dates not in catalog. Must abstain and suggest checking registrar website."},

    {"id": 25, "cat": "trick", "q": "Is there a waitlist system for oversubscribed courses at MIT?",
     "completed": [], "expect": "abstain",
     "rubric": "Waitlist info not in catalog. Must abstain and suggest checking registrar/department."}
]

print(f"✅ Test suite loaded: {len(test_suite)} queries")
print(f"   Prereq checks: {sum(1 for t in test_suite if t['cat']=='prereq')}")
print(f"   Chain questions: {sum(1 for t in test_suite if t['cat']=='chain')}")
print(f"   Program questions: {sum(1 for t in test_suite if t['cat']=='program')}")
print(f"   Trick/abstain: {sum(1 for t in test_suite if t['cat']=='trick')}")


✅ Test suite loaded: 25 queries
   Prereq checks: 10
   Chain questions: 5
   Program questions: 5
   Trick/abstain: 5


In [91]:
# ══════════════════════════════════════════════════════
# RUN EVALUATION
# ══════════════════════════════════════════════════════
eval_results = []

for t in test_suite:
    print(f"\n{'='*60}")
    print(f"[{t['id']}/{len(test_suite)}] ({t['cat']}) {t['q'][:55]}...")
    print(f"  Expected: {t['expect']}")
    print(f"  Rubric: {t['rubric'][:80]}...")

    try:
        # Set up profile based on query type
        major = "Computer Science"
        result = run_query(
            t["q"],
            completed=t.get("completed", []),
            major=major,
            term="Fall",
            max_credits=4
        )

        answer = result.get("answer", "") or ""
        decision = result.get("decision", "") or ""

        # ── METRIC 1: Citation present? ──
        has_citation = bool(re.search(r'chunk_\d|Source:|\.txt|\(chunk_', answer, re.I))

        # ── METRIC 2: Eligibility correctness (prereq queries only) ──
        eligibility_correct = None
        if t["cat"] == "prereq":
            expected = t["expect"].lower().replace(" ", "")
            actual = decision.lower().replace(" ", "").replace("n/a", "")
            eligibility_correct = expected == actual


        # ── METRIC 3: Abstention accuracy (trick queries only) ──
        abstains = None
        if t["cat"] == "trick":
            abstains = bool(re.search(
                r"don.t have that info|not in.*(catalog|document|provided)|cannot (find|determine|verify)|not available in",
                answer, re.I
            ))

        # ── METRIC 4: Structured format? ──
        has_format = bool(re.search(r'Answer\s*/?\s*Plan|Why.*prereq|Citations', answer, re.I))

        eval_results.append({
            "ID": t["id"],
            "Category": t["cat"],
            "Query": t["q"][:45],
            "Expected": t["expect"],
            "Got Decision": decision,
            "Citation": "✅" if has_citation else "❌",
            "Correct": "✅" if eligibility_correct else ("❌" if eligibility_correct is False else "—"),
            "Abstains": "✅" if abstains else ("❌" if abstains is False else "—"),
            "Format": "✅" if has_format else "❌",
            "Full Answer": answer
        })

        print(f"  → Decision: {decision} | Citation: {'✅' if has_citation else '❌'} | "
              f"Correct: {'✅' if eligibility_correct else ('❌' if eligibility_correct is False else '—')} | "
              f"Abstains: {'✅' if abstains else ('❌' if abstains is False else '—')}")

    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        eval_results.append({
            "ID": t["id"], "Category": t["cat"], "Query": t["q"][:45],
            "Expected": t["expect"], "Got Decision": "ERROR",
            "Citation": "❌", "Correct": "❌", "Abstains": "❌", "Format": "❌",
            "Full Answer": str(e)
        })

print("\n\n" + "="*70)
print("EVALUATION COMPLETE")
print("="*70)



[1/25] (prereq) Can I take 6.1010 if I passed 6.1000?...
  Expected: Eligible
  Rubric: 6.1010 prereq = 6.1000 OR (6.100A AND 6.100B). Student has 6.1000 → Eligible...

🔵 [INTAKE] All fields present → proceeding

🟡 [RETRIEVER] 25 chunks (filtered from 25, deduped)
  → [course] 6.1010 Fundamentals of Programming.txt chunk_0 | score: 0.7906
  → [course] 6.1000 Introduction to Programming and Computer Science.txt chunk_0 | score: 0.7884
  → [course] 6.1100 Computer Language Engineering.txt chunk_0 | score: 0.7779
  → [course] 6.1000 Introduction to Programming and Computer Science.txt chunk_1 | score: 0.7655
  → [course] 6.100A Introduction to Computer Science Programming in Python.txt chunk_0 | score: 0.7609
  → [course] 6.100L Introduction to Computer Science and Programming.txt chunk_0 | score: 0.7572
  → [course] 6.1020 Software Construction.txt chunk_0 | score: 0.7531
  → [course] 6.1120 Dynamic Computer Language Engineering.txt chunk_0 | score: 0.7469
  → [course] 6.1060 Software P

In [92]:
# ══════════════════════════════════════════════════════
# METRICS REPORT
# ══════════════════════════════════════════════════════
import pandas as pd

df = pd.DataFrame(eval_results)

total = len(df)
citation_count = (df["Citation"] == "✅").sum()
citation_rate = citation_count / total * 100

prereq_df = df[df["Category"] == "prereq"]
prereq_correct = (prereq_df["Correct"] == "✅").sum()
prereq_total = len(prereq_df)
prereq_acc = prereq_correct / prereq_total * 100 if prereq_total > 0 else 0

trick_df = df[df["Category"] == "trick"]
trick_abstain = (trick_df["Abstains"] == "✅").sum()
trick_total = len(trick_df)
abstain_acc = trick_abstain / trick_total * 100 if trick_total > 0 else 0

format_count = (df["Format"] == "✅").sum()
format_rate = format_count / total * 100

print("╔══════════════════════════════════════════════════╗")
print("║         EVALUATION METRICS REPORT               ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Citation Coverage Rate:    {citation_rate:5.1f}% ({citation_count}/{total})     ║")
print(f"║  Eligibility Correctness:   {prereq_acc:5.1f}% ({prereq_correct}/{prereq_total})      ║")
print(f"║  Abstention Accuracy:       {abstain_acc:5.1f}% ({trick_abstain}/{trick_total})       ║")
print(f"║  Structured Format Rate:    {format_rate:5.1f}% ({format_count}/{total})     ║")
print("╚══════════════════════════════════════════════════╝")

print("\n── Grading Rubric ──")
print("Eligibility: Checked by comparing deterministic DECISION vs expected.")
print("  Eligible = prereqs satisfied (AND: all present, OR: one present, MIXED: any branch)")
print("  Not Eligible = at least one prereq missing")
print("Citations: Regex check for (chunk_X) or .txt reference in response")
print("Abstention: Regex for 'don't have that info' / 'not in catalog' phrases")

print("\n── Per-Category Breakdown ──")
for cat in ["prereq", "chain", "program", "trick"]:
    sub = df[df["Category"] == cat]
    c_rate = (sub["Citation"] == "✅").sum() / len(sub) * 100
    f_rate = (sub["Format"] == "✅").sum() / len(sub) * 100
    print(f"  {cat.upper():10s} | Citations: {c_rate:.0f}% | Format: {f_rate:.0f}% | Count: {len(sub)}")

print("\n── Results Table ──")
display(df[["ID","Category","Query","Expected","Got Decision","Citation","Correct","Abstains","Format"]])


╔══════════════════════════════════════════════════╗
║         EVALUATION METRICS REPORT               ║
╠══════════════════════════════════════════════════╣
║  Citation Coverage Rate:     96.0% (24/25)     ║
║  Eligibility Correctness:   100.0% (10/10)      ║
║  Abstention Accuracy:       100.0% (5/5)       ║
║  Structured Format Rate:    100.0% (25/25)     ║
╚══════════════════════════════════════════════════╝

── Grading Rubric ──
Eligibility: Checked by comparing deterministic DECISION vs expected.
  Eligible = prereqs satisfied (AND: all present, OR: one present, MIXED: any branch)
  Not Eligible = at least one prereq missing
Citations: Regex check for (chunk_X) or .txt reference in response
Abstention: Regex for 'don't have that info' / 'not in catalog' phrases

── Per-Category Breakdown ──
  PREREQ     | Citations: 100% | Format: 100% | Count: 10
  CHAIN      | Citations: 80% | Format: 100% | Count: 5
  PROGRAM    | Citations: 100% | Format: 100% | Count: 5
  TRICK      | Citati

,ID,Category,Query,Expected,Got Decision,Citation,Correct,Abstains,Format
0,1,prereq,Can I take 6.1010 if I passed 6.1000?,Eligible,Eligible,✅,✅,—,✅
1,2,prereq,I have taken 6.100A and 6.100B. Can I enroll,Eligible,Eligible,✅,✅,—,✅
2,3,prereq,Can I take 6.1010 without any prior courses?,Not Eligible,Not Eligible,✅,✅,—,✅
3,4,prereq,I completed only 6.100A but not 6.100B. Am I,Not Eligible,Not Eligible,✅,✅,—,✅
4,5,prereq,I have 6.1010 completed. Am I eligible for 6.,Eligible,Eligible,✅,✅,—,✅
5,6,prereq,Can I take 6.1100 if I only have 6.1020?,Not Eligible,Not Eligible,✅,✅,—,✅
6,7,prereq,I have 6.1020 and 6.1910. Can I take 6.1100?,Eligible,Eligible,✅,✅,—,✅
7,8,prereq,Is 6.100A alone sufficient to enroll in 6.101,Not Eligible,Not Eligible,✅,✅,—,✅
8,9,prereq,Can I take 6.1120 if I have completed 6.1020?,Eligible,Eligible,✅,✅,—,✅
9,10,prereq,Can I take 6.1120 without any courses complet,Not Eligible,Not Eligible,✅,✅,—,✅


In [70]:
# ══════════════════════════════════════════════════════
# 3 EXAMPLE TRANSCRIPTS (Required by Assignment)
# ══════════════════════════════════════════════════════

print("=" * 70)
print("TRANSCRIPT 1: Correct Eligibility Decision with Citations")
print("=" * 70)
t1 = run_query(
    "Can I take 6.1010 Fundamentals of Programming if I have completed 6.100A and 6.100B?",
    completed=["6.100A", "6.100B"],
    major="Computer Science"
)
print("\n📋 FULL RESPONSE:")
print(t1["answer"])
print(f"\n🔍 DECISION: {t1['decision']}")
print(f"📌 EXPECTED: Eligible (6.100A AND 6.100B satisfies the AND branch of the OR prerequisite)")

print("\n\n" + "=" * 70)
print("TRANSCRIPT 2: Course Plan Output with Justification + Citations")
print("=" * 70)
t2 = run_query(
    "I want to plan my courses for next semester. What should I take?",
    completed=["6.1000", "6.1010", "6.1200"],
    major="Computer Science and Engineering",
    term="Spring",
    max_credits=4,
    grades={"6.1000": "A", "6.1010": "B+", "6.1200": "A-"}
)
print("\n📋 FULL RESPONSE:")
print(t2["answer"])
print(f"\n🔍 DECISION: {t2['decision']}")
print(f"📌 EXPECTED: Eligible courses like 6.1020, 6.1120, etc. with justification per course")

print("\n\n" + "=" * 70)
print("TRANSCRIPT 3: Correct Abstention + Guidance")
print("=" * 70)
t3 = run_query(
    "What is the waitlist policy for 6.1010? Is it offered on Tuesdays and Thursdays?",
    completed=[],
    major="Computer Science"
)
print("\n📋 FULL RESPONSE:")
print(t3["answer"])
print(f"\n🔍 DECISION: {t3['decision']}")
print(f"📌 EXPECTED: Abstention — waitlist policy and day schedule are NOT in the catalog documents")
print(f"📌 SHOULD SUGGEST: Check registrar website, schedule of classes, or contact advisor")

print("\n\n" + "=" * 70)
print("ALL 3 TRANSCRIPTS COMPLETE")
print("=" * 70)


TRANSCRIPT 1: Correct Eligibility Decision with Citations

🔵 [INTAKE] All fields present

🟡 [RETRIEVER] 25 chunks (filtered from 25, deduped)
  → [course] 6.1000 Introduction to Programming and Computer Science.txt chunk_1 | score: 0.799
  → [course] 6.100A Introduction to Computer Science Programming in Python.txt chunk_0 | score: 0.7923
  → [course] 6.1010 Fundamentals of Programming.txt chunk_0 | score: 0.7787
  → [course] 6.100B Introduction to Computational Thinking and Data Science.txt chunk_0 | score: 0.7699
  → [course] 6.1000 Introduction to Programming and Computer Science.txt chunk_0 | score: 0.7606
  → [course] 6.1100 Computer Language Engineering.txt chunk_0 | score: 0.7478
  → [course] 6.100L Introduction to Computer Science and Programming.txt chunk_0 | score: 0.741
  → [programs] cs_engineering_program.txt chunk_1 | score: 0.7385
  → [course] 6.1020 Software Construction.txt chunk_0 | score: 0.729
  → [course] 6.5150 Large-scale Symbolic Systems.txt chunk_0 | score: 0.7